# Human-in-the-Loop

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) LangChain roadmap.

You will learn how to pause agents for human approval before executing sensitive actions.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define a Sensitive Tool

This tool performs an irreversible action that should require human approval.

In [ ]:
from langchain_core.tools import tool

@tool
def refund_payment(order_id: str, amount: float) -> str:
    """Process a refund for an order. This action cannot be undone."""
    return f"Refund of ${amount:.2f} processed for order {order_id}"

## 4. Create an Agent with a Checkpointer

A checkpointer is required to save state at the interrupt point.

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

model = init_chat_model("gpt-4o-mini", model_provider="openai")
checkpointer = InMemorySaver()

agent = create_react_agent(
    model=model,
    tools=[refund_payment],
    checkpointer=checkpointer,
)

## 5. Interrupt Before Tool Execution

Use `interrupt_before=["tools"]` to pause the agent before it calls a tool.

In [ ]:
config = {"configurable": {"thread_id": "refund-1"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Refund $50 for order ORD-123")]},
    config=config,
    interrupt_before=["tools"],
)

print("Agent paused. Inspecting pending action...")

## 6. Inspect the Pending Action

In [ ]:
state = agent.get_state(config)

for msg in state.values["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"Tool: {tc['name']}")
            print(f"Args: {tc['args']}")

## 7. Approve — Resume Execution

Pass `Command(resume=True)` to approve the pending action.

In [ ]:
from langgraph.types import Command

result = agent.invoke(Command(resume=True), config=config)
print("Approved:", result["messages"][-1].content)

## 8. Reject — Block the Action

Start a new thread and reject the action by passing a string message.

In [ ]:
config_2 = {"configurable": {"thread_id": "refund-2"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Refund $200 for order ORD-789")]},
    config=config_2,
    interrupt_before=["tools"],
)

print("Agent paused. Rejecting...")

result = agent.invoke(
    Command(resume="Refund denied. The amount exceeds the policy limit. Suggest store credit."),
    config=config_2,
)
print("Rejected:", result["messages"][-1].content)

## 9. Edit — Modify the Action

Resume with new arguments to change the tool call.

In [ ]:
config_3 = {"configurable": {"thread_id": "refund-3"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Refund $100 for order ORD-456")]},
    config=config_3,
    interrupt_before=["tools"],
)

print("Agent paused. Editing amount to $75...")

result = agent.invoke(
    Command(resume={"order_id": "ORD-456", "amount": 75.00}),
    config=config_3,
)
print("Edited:", result["messages"][-1].content)

## Summary

- Use `interrupt_before=["tools"]` to pause before tool execution
- A checkpointer (`InMemorySaver`) is required for HITL
- Approve with `Command(resume=True)`
- Reject with `Command(resume="reason")`
- Edit with `Command(resume={...new_args})`
- Inspect pending actions via `agent.get_state(config)`